In [1]:
import pandas as pd
from dotenv import load_dotenv
import os

In [2]:
#läser in datasetet från MovieLens
#movieLens använder "," som separator
df_movies = pd.read_csv("dataset/movies.csv", sep=",")
df_links = pd.read_csv("dataset/links.csv", sep=",")
df_ratings = pd.read_csv("dataset/ratings.csv", sep=",")

In [3]:
#visar de 5 första raderna för att få överblick över datan
print(df_movies.head())

#visar hur många kolumner och rader datasetet innehåller
print(df_movies.shape)

#visar kolumnerna/features
print(df_movies.columns)

#kollar hur många dubletter det finns i datasetet
df_movies.duplicated().sum()

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
(9742, 3)
Index(['movieId', 'title', 'genres'], dtype='str')


np.int64(0)

In [4]:
#visar de 5 första raderna för att få överblick över datan
print(df_links.head())

#visar hur många kolumner och rader datasetet innehåller
print(df_links.shape)

#visar kolumnerna/features
print(df_links.columns)

#kollar hur många dubletter det finns i datasetet
df_links.duplicated().sum()

   movieId  imdbId   tmdbId
0        1  114709    862.0
1        2  113497   8844.0
2        3  113228  15602.0
3        4  114885  31357.0
4        5  113041  11862.0
(9742, 3)
Index(['movieId', 'imdbId', 'tmdbId'], dtype='str')


np.int64(0)

In [5]:
#visar de 5 första raderna för att få överblick över datan
print(df_ratings.head())

#visar hur många kolumner och rader datasetet innehåller
print(df_ratings.shape)

#visar kolumnerna/features
print(df_ratings.columns)

#kollar hur många dubletter det finns i datasetet
df_ratings.duplicated().sum()

   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931
(100836, 4)
Index(['userId', 'movieId', 'rating', 'timestamp'], dtype='str')


np.int64(0)

In [6]:
#grupperar alla ratings efter movieId och räknar ut genomsnittsbetyget för varje film
avg_ratings = df_ratings.groupby("movieId")["rating"].mean().reset_index()

#byter namn på kolumnen från rating till movielens_avg_rating för att göra det tydligare att det är MovieLens genomsnittsbetyg
avg_ratings = avg_ratings.rename(columns={"rating": "movielens_avg_rating"})

In [7]:
#droppar kolumnen "imdbId" från df_links eftersom vi endast behöver tmdbId och inte imdbId
df_links = df_links.drop(columns=["imdbId"])

In [8]:
#Merga df_links och till df_movies med movieId som gemensam nyckel och lägger till avg_ratings som en ny kolumn/feature
df = df_movies.merge(df_links, on="movieId")
df = df.merge(avg_ratings, on="movieId")

In [9]:
#visar hur många kolumner och rader datasetet innehåller
print(df.shape)
#visar datasetets struktur
print(df.info())

(9724, 5)
<class 'pandas.DataFrame'>
RangeIndex: 9724 entries, 0 to 9723
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   movieId               9724 non-null   int64  
 1   title                 9724 non-null   str    
 2   genres                9724 non-null   str    
 3   tmdbId                9716 non-null   float64
 4   movielens_avg_rating  9724 non-null   float64
dtypes: float64(2), int64(1), str(2)
memory usage: 380.0 KB
None


In [10]:
import requests
import time

#laddar miljövariabler från .env-filen
load_dotenv()

#hämtar TMDb API-nyckeln från .env-filen
#API-nyckel för autentisering mot TMDb API
API_KEY = os.getenv("TMDB_API_KEY")


#funktion som hämtar filmdata från TMDb API baserat på tmdbId
def get_tmdb_data(tmdbId):

    #kontroll av saknade värden
    if pd.isna(tmdbId):
        return None
    
    #skapar URL till rätt film i TMDb API
    url = f"https://api.themoviedb.org/3/movie/{int(tmdbId)}"

    #skickar API-nyckeln som parameter till API-anropet
    params = {
        "api_key": API_KEY
    }

    #skickar GET-request till TMDb API
    response = requests.get(url, params=params)

    #kontrollerar om API-anropet lyckades
    if response.status_code !=200:
        return None
    
    #konverterar API-svaret från JSON till Python-dictionary
    data = response.json()

    #returnerar utvalda features från TMDb-datan
    return {
        #filmens beskrivning/sammanfattning
        "overview": data.get("overview"),
        #sökväg till filmens posterbild
        "poster_path": data.get("poster_path"),
        #filmens releasedatum
        "release_date": data.get("release_date"),
        #filmens längd i minuter
        "runtime": data.get("runtime"),
        #TMDb:s popularitetspoäng
        "popularity": data.get("popularity")
    }

#tom lista som används för att lagra filmdata
rows = []

#loopar igenom filmerna i datasetet
for index, row in df.iterrows():
#for index, row in df.head(500).iterrows():

    #hämtar TMDb-data för aktuell film
    tmdb_data = get_tmdb_data(row["tmdbId"])

    #kontrollerar att TMDb-data finns innan den sparas
    if tmdb_data is not None:
        rows.append({
            "movieId": row["movieId"],
            "title": row["title"],
            "genres": row["genres"],
            "tmdbId": row["tmdbId"],
            "movielens_avg_rating": row["movielens_avg_rating"],
            **tmdb_data
        })

    #väntar mellan API-anrop för att undvika för många requests mot TMDb API
    time.sleep(0.25)

#konverterar listan rows till en pandas DataFrame
df_cleaned = pd.DataFrame(rows)

#sparar den färdiga dataframe:n som CSV-fil
df_cleaned.to_csv("dataset/movies_cleaned.csv", index=False)